# Vẽ một EEM trong không gian 3 chiều

Notebook đọc một mẫu từ `data/processed/eem.npy` và vẽ bề mặt với **X = Ex (nm)**, **Y = Em (nm)**, **Z = Intensity**. Màu sắc cũng biểu diễn Intensity.

Chọn môi trường Python của dự án và **Run All**. Nếu thiếu thư viện, cài `python -m pip install -e ".[notebooks]"` tại thư mục gốc dự án.

Đây là dữ liệu đã xử lý; notebook giữ nguyên thang cường độ, không chuẩn hóa hay biến đổi log. Đơn vị cường độ không được khai báo trong tệp dữ liệu nên trục Z không gán đơn vị.

Biểu đồ Plotly tương tác: kéo chuột để xoay, cuộn để phóng to/thu nhỏ, rê chuột để xem giá trị. Thanh công cụ hỗ trợ di chuyển và đặt lại góc nhìn. Cell cuối lưu bản HTML độc lập để mở bằng trình duyệt.

**Phần heatmap PARAFAC:** sau mục “Phân rã toàn bộ EEM và xem phổ thành phần”, xem mục **“Heatmap component và đóng góp vào một EEM”** (trước Pearson r). Chạy hai cell code trong mục này để hiển thị hình; đổi `HEATMAP_SAMPLE_INDEX` để chọn mẫu. Các cell mới chưa có output lưu sẵn.


Nếu kernel thiếu `nbformat`, hàm `show_plotly` tự chuyển sang HTML tương tác. Để dùng renderer gốc, chạy `%pip install "nbformat>=4.2.0"` trong một cell của chính notebook rồi restart kernel. Sau khi cập nhật notebook, chạy lại cell import để nạp hàm hiển thị.


Benchmark phân rã/hồi quy riêng ở cuối notebook dùng `Group` từ processed data, gồm mọi nhóm có ≥8 mẫu. `single_*` là từng nhóm riêng. CV fit decomposition riêng trên train của từng fold; đây là shuffled CV, không phải temporal CV. Bảng best single component vẫn là lựa chọn khám phá trên cùng CV.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from IPython.display import HTML, display

# Tìm dữ liệu khi chạy từ thư mục gốc hoặc thư mục notebooks.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "processed" / "eem.npy").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook trong dự án có data/processed/eem.npy.")

DATA_DIR = ROOT / "data" / "processed"
eems = np.load(DATA_DIR / "eem.npy", mmap_mode="r")
samples = pd.read_csv(DATA_DIR / "samples.csv")
with np.load(DATA_DIR / "wavelengths.npz") as wavelengths:
    ex = wavelengths["excitation"]
    em = wavelengths["emission"]

assert eems.shape == (len(samples), len(em), len(ex)), "Kích thước dữ liệu không khớp."
print(f"Số mẫu: {len(eems)} | Mỗi EEM: {len(em)} Em × {len(ex)} Ex")
print(f"Ex: {ex.min():g}–{ex.max():g} nm | Em: {em.min():g}–{em.max():g} nm")

def show_plotly(figure, config=None):
    """Use notebook MIME rendering, with inline HTML if nbformat is unavailable."""
    plot_config = {"scrollZoom": True, "displaylogo": False, "responsive": True}
    if config is not None:
        plot_config.update(config)
    try:
        figure.show(renderer="plotly_mimetype", config=plot_config)
    except ValueError as error:
        if "nbformat" not in str(error):
            raise
        # Inline Plotly JS keeps the fallback usable without a network connection.
        display(HTML(figure.to_html(
            full_html=False, include_plotlyjs=True, config=plot_config,
        )))


## Chọn mẫu

Giữ `SAMPLE_INDEX = None` để chọn ngẫu nhiên với `SEED` cố định (kết quả lặp lại được). Đổi seed để chọn mẫu khác, hoặc đặt `SAMPLE_INDEX` từ `0` đến `759` để chọn trực tiếp theo vị trí hàng.

In [ ]:
SEED = 42
SAMPLE_INDEX = None

if SAMPLE_INDEX is None:
    sample_index = int(np.random.default_rng(SEED).integers(len(eems)))
else:
    if not isinstance(SAMPLE_INDEX, (int, np.integer)) or not 0 <= SAMPLE_INDEX < len(eems):
        raise ValueError(f"SAMPLE_INDEX phải là số nguyên từ 0 đến {len(eems) - 1}.")
    sample_index = int(SAMPLE_INDEX)

eem = np.asarray(eems[sample_index])
metadata = samples.iloc[sample_index]
fields = ["sample_id", "eem_filename", "Point", "Year", "Month", "Round", "Depth"]
print(f"Vị trí mẫu trong mảng: {sample_index}")
display(metadata[[field for field in fields if field in samples.columns]].to_frame("Giá trị"))
finite = eem[np.isfinite(eem)]
if finite.size == 0:
    raise ValueError("Mẫu được chọn không có cường độ hữu hạn để vẽ.")
print(f"Intensity: min = {finite.min():.3f}, max = {finite.max():.3f}")

In [ ]:
# # EEM có thứ tự (Em, Ex): hàng ứng với Y, cột ứng với X.
# assert eem.shape == (len(em), len(ex))

# fig = go.Figure(go.Surface(
#     x=ex,
#     y=em,
#     z=np.where(np.isfinite(eem), eem, np.nan),
#     colorscale="Viridis",
#     colorbar={"title": "Intensity"},
#     connectgaps=False,
#     hovertemplate=(
#         "Ex: %{x:.1f} nm<br>"
#         "Em: %{y:.1f} nm<br>"
#         "Intensity: %{z:.3f}<extra></extra>"
#     ),
# ))
# fig.update_layout(
#     title=f"EEM #{sample_index} | {metadata['eem_filename']}<br>{metadata['Point']}",
#     template="plotly_white",
#     height=750,
#     margin=dict(l=0, r=0, b=0, t=80),
#     scene=dict(
#         xaxis_title="Ex (nm)",
#         yaxis_title="Em (nm)",
#         zaxis_title="Intensity",
#         aspectmode="manual",
#         aspectratio=dict(x=1, y=1, z=0.7),
#         camera=dict(eye=dict(x=1.5, y=1.5, z=1.1)),
#     ),
# )
# config = {"scrollZoom": True, "displaylogo": False, "responsive": True}
# show_plotly(fig, config=config)

In [ ]:
# # HTML chứa sẵn Plotly: mở và tương tác mà không cần kết nối mạng.
# OUTPUT_DIR = ROOT / "runs" / "diagnostics" / "eem_3d"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# html_path = OUTPUT_DIR / f"eem_{sample_index}_3d.html"
# fig.write_html(html_path, include_plotlyjs=True, full_html=True, config=config)
# print(f"Đã lưu biểu đồ tương tác: {html_path}")

## PARAFAC riêng theo group

Phần này thực hiện một pipeline độc lập cho từng nhóm:

1. Đếm số EEM theo `Group` và chọn các nhóm đủ dữ liệu có nhiều mẫu nhất; khi bằng nhau, sắp xếp theo tên nhóm để kết quả tái lập.
2. Với mỗi nhóm, quét `GROUP_RANKS = [3, 4]` bằng multistart PARAFAC trên toàn bộ EEM của nhóm.
3. Kiểm tra độ ổn định bằng nhiều khởi tạo và split-half trên chính nhóm đó.
4. Chọn rank nhỏ nhất trong nhóm ổn định có `fit_energy` gần tốt nhất; nếu chưa có rank đạt ngưỡng, dùng điểm gấp khúc tạm thời và ghi trạng thái cảnh báo.
5. Chuẩn hóa score theo Fmax, sắp component theo đỉnh Em rồi Ex trong từng nhóm, sau đó tính Pearson, Spearman và (R^2) linear với BOD/COD/TOC.

Component của các nhóm được đánh số theo thứ tự phổ **trong từng nhóm**. Vì rank có thể khác nhau và PARAFAC có bất định hoán vị component, không được mặc định xem `C1` ở hai nhóm là cùng một fluorophore. Kết quả này phù hợp để nghiên cứu động lực nội nhóm; mô hình toàn bộ dữ liệu vẫn nên dùng cho so sánh chung và dự báo giữa các nhóm.

Các file của pipeline này được lưu trong `runs/diagnostics/parafac/by_group/` để không ghi đè kết quả phân tích theo nhóm trước đó.

In [ ]:
# Standalone setup: this cell does not depend on the global PARAFAC section.
from pathlib import Path
import json
import time
import warnings
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from scipy.optimize import linear_sum_assignment
from scipy.stats import pearsonr, spearmanr
from tensorly.decomposition import non_negative_parafac_hals
from tensorly.cp_tensor import cp_to_tensor
from threadpoolctl import threadpool_limits
import plotly.graph_objects as go

ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "data" / "processed" / "eem.npy").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy data/processed/eem.npy.")
DATA_DIR = ROOT / "data" / "processed"
eems = np.load(DATA_DIR / "eem.npy", mmap_mode="r")
samples = pd.read_csv(DATA_DIR / "samples.csv")
with np.load(DATA_DIR / "wavelengths.npz") as wavelengths:
    ex = wavelengths["excitation"]
    em = wavelengths["emission"]
if eems.shape != (len(samples), len(em), len(ex)):
    raise ValueError("Kích thước EEM, metadata và wavelength không khớp.")

X_raw = np.asarray(eems, dtype=np.float64)
if not np.isfinite(X_raw).all():
    raise ValueError("EEM có NaN/Inf.")
X_raw = np.maximum(X_raw, 0)
scale = np.linalg.norm(X_raw)
if scale <= 0:
    raise ValueError("Tensor EEM toàn bằng 0.")
X = X_raw / scale

group_targets = pd.DataFrame(index=samples.index)
for target in ["BOD", "COD", "TOC"]:
    candidates = [
        column for column in samples.columns
        if column.strip() == target or column.splitlines()[0].strip() == target
    ]
    if len(candidates) != 1:
        raise ValueError(f"Không tìm thấy duy nhất cột {target}: {candidates}")
    group_targets[target] = pd.to_numeric(samples[candidates[0]], errors="coerce")

GROUP_SEED = 42
GROUP_SIMILARITY_THRESHOLD = 0.95
GROUP_FIT_TOLERANCE = 0.01


def group_show_plotly(figure):
    config = {"scrollZoom": True, "displaylogo": False, "responsive": True}
    try:
        figure.show(renderer="plotly_mimetype", config=config)
    except ValueError as error:
        if "nbformat" not in str(error):
            raise
        display(HTML(figure.to_html(full_html=False, include_plotlyjs=True, config=config)))


from scipy.stats import pearsonr, spearmanr

FIT_BY_GROUP = True
TOP_GROUP_COUNT = None  # All groups with at least 8 samples
GROUP_RANKS = [3, 4]  # Benchmark chỉ fit hai ứng viên rank 3 và rank 4.
GROUP_N_STARTS = 3
GROUP_SPLIT_REPEATS = 2
GROUP_MAX_ITER = 500
GROUP_TOL = 1e-7
GROUP_MIN_SAMPLES = 30
GROUP_OUTPUT = ROOT / "runs" / "diagnostics" / "parafac" / "by_group"
GROUP_OUTPUT.mkdir(parents=True, exist_ok=True)


def fit_group_multistart(tensor, rank, seed):
    fits = []
    with threadpool_limits(limits=2):
        for start in range(GROUP_N_STARTS):
            kwargs = dict(
                tensor=tensor,
                rank=rank,
                init="random",
                random_state=seed + start,
                n_iter_max=GROUP_MAX_ITER,
                tol=GROUP_TOL,
                return_errors=True,
                normalize_factors=False,
            )
            cp, errors = non_negative_parafac_hals(**kwargs)
            reconstructed = cp_to_tensor(cp)
            error = float(
                np.linalg.norm(tensor - reconstructed)
                / max(np.linalg.norm(tensor), 1e-15)
            )
            converged = len(errors) >= 2 and abs(float(errors[-1] - errors[-2])) < GROUP_TOL
            fits.append({
                "cp": cp,
                "error": error,
                "iterations": len(errors),
                "converged": bool(converged),
                "start": start,
            })
    return min(fits, key=lambda item: item["error"]), fits


def match_group_spectra(left, right):
    similarities = []
    for mode in (1, 2):
        left_factor = left.factors[mode]
        right_factor = right.factors[mode]
        left_factor = left_factor / np.maximum(
            np.linalg.norm(left_factor, axis=0, keepdims=True), 1e-15
        )
        right_factor = right_factor / np.maximum(
            np.linalg.norm(right_factor, axis=0, keepdims=True), 1e-15
        )
        similarities.append(np.clip(left_factor.T @ right_factor, 0, 1))
    joint = np.minimum(*similarities)
    rows, cols = linear_sum_assignment(-joint)
    return float(joint[rows, cols].min())


def fit_one_group_rank(tensor, rank, seed):
    full_best, full_starts = fit_group_multistart(tensor, rank, seed)
    rng = np.random.default_rng(seed + 900000)
    split_rows = []
    for repeat, halves in enumerate(
        [np.array_split(rng.permutation(len(tensor)), 2) for _ in range(GROUP_SPLIT_REPEATS)],
        start=1,
    ):
        half_models = []
        for half, indices in enumerate(halves):
            half_best, _ = fit_group_multistart(
                tensor[indices],
                rank,
                seed + 10000 * repeat + 100 * (half + 1),
            )
            half_models.append(half_best)
        similarity = min(
            match_group_spectra(half_models[0]["cp"], half_models[1]["cp"]),
            *(match_group_spectra(full_best["cp"], half["cp"]) for half in half_models),
        )
        split_rows.append({
            "repeat": repeat,
            "similarity": similarity,
            "converged": all(half["converged"] for half in half_models),
        })
    start_similarity = min(
        match_group_spectra(full_best["cp"], fit["cp"])
        for fit in full_starts
    )
    split_similarity = min(row["similarity"] for row in split_rows)
    return {
        "best": full_best,
        "starts": full_starts,
        "rank": rank,
        "relative_error": full_best["error"],
        "fit_energy": 1 - full_best["error"] ** 2,
        "start_similarity": start_similarity,
        "split_similarity": split_similarity,
        "converged": full_best["converged"] and all(row["converged"] for row in split_rows),
        "converged_starts": sum(fit["converged"] for fit in full_starts),
    }


def choose_group_rank(rank_table):
    stable = rank_table[
        rank_table["converged"]
        & (rank_table["start_similarity"] >= GROUP_SIMILARITY_THRESHOLD)
        & (rank_table["split_similarity"] >= GROUP_SIMILARITY_THRESHOLD)
    ]
    if len(stable):
        near_best = stable[
            stable["fit_energy"] >= stable["fit_energy"].max() - GROUP_FIT_TOLERANCE
        ]
        return int(near_best["rank"].min()), "stable_near_best"
    errors = rank_table["relative_error"].to_numpy()
    positions = np.linspace(0, 1, len(errors))
    improvement = (errors[0] - errors) / max(errors[0] - errors[-1], 1e-15)
    proposed = int(rank_table.iloc[np.argmax(improvement - positions)]["rank"])
    return proposed, "temporary_elbow_no_stable_rank"


def normalize_group_cp(cp, intensity_scale):
    scores_, emission_, excitation_ = [
        np.array(factor, copy=True, dtype=float) for factor in cp.factors
    ]
    weights_ = np.ones(scores_.shape[1]) if cp.weights is None else np.asarray(cp.weights, dtype=float)
    emission_peak = emission_.max(axis=0)
    excitation_peak = excitation_.max(axis=0)
    if np.any(emission_peak <= 0) or np.any(excitation_peak <= 0):
        raise ValueError("Có component suy biến trong fit theo nhóm.")
    scores_ *= weights_ * emission_peak * excitation_peak * intensity_scale
    emission_ /= emission_peak
    excitation_ /= excitation_peak
    order = np.lexsort((ex[excitation_.argmax(axis=0)], em[emission_.argmax(axis=0)]))
    return scores_[:, order], emission_[:, order], excitation_[:, order]


def group_metric_row(group, selected_rank_, component, target, x, y):
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = np.asarray(x[valid], dtype=float), np.asarray(y[valid], dtype=float)
    row = {
        "Group": group,
        "selected_rank": selected_rank_,
        "component": component,
        "target": target,
        "n": int(len(x)),
        "pearson_r": np.nan,
        "spearman_rho": np.nan,
        "R2_linear": np.nan,
        "status": "ok",
    }
    if len(x) < 5 or np.ptp(x) <= 0 or np.ptp(y) <= 0:
        row["status"] = "insufficient_or_constant"
        return row
    row["pearson_r"] = float(pearsonr(x, y).statistic)
    row["spearman_rho"] = float(spearmanr(x, y).statistic)
    slope, intercept = np.polyfit(x, y, 1)
    prediction = intercept + slope * x
    sse = float(np.sum((y - prediction) ** 2))
    sst = float(np.sum((y - y.mean()) ** 2))
    row["R2_linear"] = float(1 - sse / sst) if sst > 0 else np.nan
    return row


if FIT_BY_GROUP:
    group_counts = (
        samples.groupby("Group", as_index=False)
        .size()
        .rename(columns={"size": "n_samples"})
        .sort_values(["n_samples", "Group"], ascending=[False, True], kind="mergesort")
    )
    selected_groups = group_counts[group_counts.n_samples >= GROUP_MIN_SAMPLES].head(TOP_GROUP_COUNT).reset_index(drop=True)
    selected_group_names = selected_groups["Group"].tolist()
    if TOP_GROUP_COUNT is not None and len(selected_groups) < TOP_GROUP_COUNT:
        warnings.warn("Số nhóm thực tế nhỏ hơn TOP_GROUP_COUNT.")
    if selected_groups["n_samples"].min() < GROUP_MIN_SAMPLES:
        warnings.warn("Có nhóm được chọn nhưng có ít mẫu hơn GROUP_MIN_SAMPLES.")
    selected_groups.to_csv(GROUP_OUTPUT / "selected_top_groups.csv", index=False)
    display(selected_groups)

    group_rank_rows = []
    group_fit_rows = []
    group_correlation_rows = []
    group_score_frames = []
    group_models = {}

    for group_position, group in enumerate(selected_group_names):
        group_started = time.perf_counter()
        indices = samples.index[samples["Group"].eq(group)].to_numpy(dtype=int)
        print(
            f"[nhóm {group_position + 1}/{len(selected_group_names)}] "
            f"{group}: {len(indices)} EEM",
            flush=True,
        )
        n_samples = len(indices)
        if n_samples < GROUP_MIN_SAMPLES:
            group_fit_rows.append({
                "Group": group,
                "n_samples": n_samples,
                "status": "skipped_too_few_samples",
            })
            print("  bỏ qua: không đủ mẫu", flush=True)
            continue
        group_tensor = X[indices]
        rank_models = {}
        for rank in GROUP_RANKS:
            rank_started = time.perf_counter()
            result = fit_one_group_rank(
                group_tensor,
                rank,
                GROUP_SEED + 100000 * (group_position + 1) + 1000 * rank,
            )
            rank_models[rank] = result
            print(
                f"  rank {rank}/{max(GROUP_RANKS)}: fit_energy={result['fit_energy']:.4f}, split_similarity={result['split_similarity']:.4f},"
                f"({time.perf_counter() - rank_started:.1f}s)",
                flush=True,
            )
            group_rank_rows.append({
                "Group": group,
                "n_samples": n_samples,
                **{key: value for key, value in result.items()
                   if key not in {"best", "starts"}},
            })
        rank_table = pd.DataFrame([
            row for row in group_rank_rows if row["Group"] == group
        ]).sort_values("rank")
        chosen_rank, selection_status = choose_group_rank(rank_table)
        chosen = rank_models[chosen_rank]
        group_models[group] = chosen
        group_fit_rows.append({
            "Group": group,
            "n_samples": n_samples,
            "status": "fit",
            "selected_rank": chosen_rank,
            "selection_status": selection_status,
            "relative_error": chosen["relative_error"],
            "fit_energy": chosen["fit_energy"],
            "start_similarity": chosen["start_similarity"],
            "split_similarity": chosen["split_similarity"],
            "converged": chosen["converged"],
        })

        group_scores, group_emission, group_excitation = normalize_group_cp(
            chosen["best"]["cp"], scale
        )
        group_component_names = [f"C{i + 1}" for i in range(chosen_rank)]
        group_score_frames.append(pd.DataFrame(
            group_scores,
            columns=group_component_names,
            index=indices,
        ).assign(
            sample_index=indices,
            sample_id=samples.loc[indices, "sample_id"].to_numpy(),
            group=group,
            selected_rank=chosen_rank,
        ))
        for component_index, component in enumerate(group_component_names):
            for target in group_targets.columns:
                group_correlation_rows.append(group_metric_row(
                    group,
                    chosen_rank,
                    component,
                    target,
                    group_scores[:, component_index],
                    group_targets.loc[indices, target].to_numpy(dtype=float),
                ))
        print(
            f"  hoàn tất {group} sau {time.perf_counter() - group_started:.1f}s; "
            f"rank chọn={chosen_rank}",
            flush=True,
        )

    group_fit_summary = pd.DataFrame(group_fit_rows).sort_values("Group").reset_index(drop=True)
    group_rank_diagnostics = pd.DataFrame(group_rank_rows).sort_values(
        ["Group", "rank"]
    ).reset_index(drop=True)
    group_correlations = pd.DataFrame(group_correlation_rows)
    group_scores = (
        pd.concat(group_score_frames, ignore_index=True)
        if group_score_frames
        else pd.DataFrame()
    )
    print(
        f"Đã fit {int((group_fit_summary['status'] == 'fit').sum())}/"
        f"{len(selected_groups)} nhóm được chọn."
    )
    display(group_fit_summary)
    display(group_rank_diagnostics)
    display(group_correlations.head(20).round(4))

    rank_fig = go.Figure()
    for group in selected_group_names:
        rows = group_rank_diagnostics[group_rank_diagnostics["Group"] == group]
        rank_fig.add_trace(go.Scatter(
            x=rows["rank"], y=rows["fit_energy"], mode="lines+markers", name=group,
        ))
    rank_fig.update_layout(
        template="plotly_white", title="Fit energy theo rank và nhóm",
        xaxis_title="Số component", yaxis_title="Fit energy", height=600,
    )
    group_show_plotly(rank_fig)
    rank_fig.write_html(GROUP_OUTPUT / "group_rank_selection.html", include_plotlyjs=True)

    if not group_correlations.empty:
        group_order = selected_group_names
        max_rank = max(GROUP_RANKS)
        all_component_names = [f"C{i + 1}" for i in range(max_rank)]
        column_order = pd.MultiIndex.from_product(
            [all_component_names, group_targets.columns], names=["component", "target"]
        )
        for metric, bounds, title in [
            ("pearson_r", (-1, 1), "Pearson r theo nhóm"),
            ("spearman_rho", (-1, 1), "Spearman rho theo nhóm"),
            ("R2_linear", (0, 1), "R² linear theo nhóm"),
        ]:
            matrix = group_correlations.pivot_table(
                index="Group", columns=["component", "target"], values=metric, aggfunc="first"
            ).reindex(index=group_order, columns=column_order)
            figure = go.Figure(go.Heatmap(
                x=[f"{component}/{target}" for component, target in matrix.columns],
                y=matrix.index,
                z=matrix.to_numpy(dtype=float),
                zmin=bounds[0], zmax=bounds[1],
                colorscale="RdBu_r" if bounds[0] < 0 else "Viridis",
                colorbar=dict(title=metric),
                hovertemplate="%{y}<br>%{x}: %{z:.3f}<extra></extra>",
            ))
            figure.update_layout(
                template="plotly_white", title=title,
                height=520, width=max(900, 75 * len(matrix.columns)),
                xaxis_title="Component / target", yaxis_title="Group",
            )
            group_show_plotly(figure)
            figure.write_html(
                GROUP_OUTPUT / f"group_{metric}_heatmap.html", include_plotlyjs=True
            )

    group_fit_summary.to_csv(GROUP_OUTPUT / "group_fit_summary.csv", index=False)
    group_rank_diagnostics.to_csv(GROUP_OUTPUT / "group_rank_diagnostics.csv", index=False)
    group_correlations.to_csv(
        GROUP_OUTPUT / "group_component_target_correlations.csv", index=False
    )
    group_scores.to_csv(GROUP_OUTPUT / "group_component_scores.csv", index=False)
    with (GROUP_OUTPUT / "config.json").open("w", encoding="utf-8") as handle:
        json.dump({
            "fit_by_group": FIT_BY_GROUP,
            "top_group_count": TOP_GROUP_COUNT,
            "selected_groups": selected_group_names,
            "candidate_ranks": GROUP_RANKS,
            "n_starts": GROUP_N_STARTS,
            "split_repeats": GROUP_SPLIT_REPEATS,
            "max_iter": GROUP_MAX_ITER,
            "tol": GROUP_TOL,
            "min_samples": GROUP_MIN_SAMPLES,
            "rank_selection": "stable_near_best_else_temporary_elbow",
            "component_order": "group_local_peak_Em_then_Ex",
        }, handle, indent=2, ensure_ascii=False)
    print(f"Đã lưu pipeline theo các nhóm đủ dữ liệu tại: {GROUP_OUTPUT}")

## Linear R² theo nhóm từ cực đại ma trận component PARAFAC

Với mỗi nhóm, mỗi EEM và mỗi component của mô hình PARAFAC riêng theo nhóm, lấy cực đại intensity của ma trận component tái tạo làm feature scalar. Sau đó fit hồi quy tuyến tính với BOD, COD và TOC. Kết quả được lưu riêng với correlation của score PARAFAC.

In [ ]:
# Group-level linear R² from max intensity of each reconstructed PARAFAC component.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

group_max_r2_rows = []
group_max_feature_rows = []
for group in selected_group_names:
    if group not in group_models:
        continue
    group_indices = samples.index[samples["Group"].eq(group)].to_numpy(dtype=int)
    group_result = group_models[group]
    selected_rank_group = int(group_result["rank"])
    group_scores_max, group_emission_max, group_excitation_max = normalize_group_cp(
        group_result["best"]["cp"], scale
    )
    group_component_maps = np.einsum(
        "jf,kf->fjk", group_emission_max, group_excitation_max
    )
    group_contributions = (
        group_scores_max[:, :, None, None]
        * group_component_maps[None, :, :, :]
    )
    group_max_values = np.max(
        group_contributions.reshape(len(group_indices), selected_rank_group, -1),
        axis=2,
    )
    component_names_group_max = [f"C{i + 1}" for i in range(selected_rank_group)]
    for local_index, sample_index in enumerate(group_indices):
        row = {
            "Group": group,
            "sample_index": int(sample_index),
            "sample_id": samples.loc[sample_index, "sample_id"],
            "selected_rank": selected_rank_group,
        }
        row.update({
            f"max_{name}": float(group_max_values[local_index, component_index])
            for component_index, name in enumerate(component_names_group_max)
        })
        group_max_feature_rows.append(row)
    for component_index, component_name in enumerate(component_names_group_max):
        x_all = group_max_values[:, component_index]
        for target in group_targets.columns:
            y_all = group_targets.loc[group_indices, target].to_numpy(dtype=float)
            valid = np.isfinite(x_all) & np.isfinite(y_all)
            x = x_all[valid].reshape(-1, 1)
            y = y_all[valid]
            row = {
                "Group": group,
                "selected_rank": selected_rank_group,
                "component": component_name,
                "feature": f"max_{component_name}",
                "target": target,
                "n": int(len(y)),
                "intercept": np.nan,
                "slope": np.nan,
                "R2_linear": np.nan,
                "status": "ok",
            }
            if len(y) < 5 or np.ptp(x) == 0 or np.ptp(y) == 0:
                row["status"] = "insufficient_or_constant"
            else:
                estimator = LinearRegression().fit(x, y)
                row["intercept"] = float(estimator.intercept_)
                row["slope"] = float(estimator.coef_[0])
                row["R2_linear"] = float(r2_score(y, estimator.predict(x)))
            group_max_r2_rows.append(row)

parafac_group_max_r2 = pd.DataFrame(group_max_r2_rows)
parafac_group_max_features = pd.DataFrame(group_max_feature_rows)
parafac_group_max_r2.to_csv(
    GROUP_OUTPUT / "group_component_max_linear_r2.csv", index=False
)
parafac_group_max_features.to_csv(
    GROUP_OUTPUT / "group_component_max_features.csv", index=False
)
display(parafac_group_max_r2.round(4))

max_r2_table = parafac_group_max_r2.pivot_table(
    index="Group", columns=["component", "target"], values="R2_linear", aggfunc="first"
)
all_components = [f"C{i + 1}" for i in range(max(GROUP_RANKS))]
max_r2_columns = pd.MultiIndex.from_product([all_components, ["BOD", "COD", "TOC"]])
max_r2_table = max_r2_table.reindex(index=selected_group_names, columns=max_r2_columns)
figure = go.Figure(go.Heatmap(
    x=[f"{component}/{target}" for component, target in max_r2_columns],
    y=max_r2_table.index.tolist(),
    z=max_r2_table.to_numpy(dtype=float),
    zmin=0,
    zmax=1,
    colorscale="Viridis",
    colorbar=dict(title="R²"),
    hovertemplate="%{y}<br>%{x}: %{z:.3f}<extra></extra>",
))
figure.update_layout(
    template="plotly_white",
    title="PARAFAC theo nhóm: R² từ max intensity component",
    height=max(520, 55 * len(max_r2_table)),
    width=max(1000, 75 * len(max_r2_columns)),
    xaxis_title="Component / target",
    yaxis_title="Group",
    xaxis=dict(tickangle=70),
)
group_show_plotly(figure)
figure.write_html(
    GROUP_OUTPUT / "group_component_max_R2_heatmap.html",
    include_plotlyjs=True,
)
print(f"Đã lưu group max-component R² và Plotly heatmap tại {GROUP_OUTPUT}")

## Decomposition của một EEM đại diện cho từng nhóm

Cell này chọn EEM đầu tiên theo `sample_id` ở mỗi nhóm đã chọn để tái lập kết quả. Mỗi hình gồm EEM gốc, EEM tái tạo, phần dư và heatmap đóng góp của từng component từ mô hình PARAFAC riêng của nhóm.

In [ ]:
import math
import re
import matplotlib.pyplot as plt

SHOW_GROUP_DECOMPOSITIONS = True
GROUP_DECOMP_OUTPUT = GROUP_OUTPUT / "decompositions"
GROUP_DECOMP_OUTPUT.mkdir(parents=True, exist_ok=True)


def group_file_name(value):
    return re.sub(r"[^\w.-]+", "_", str(value), flags=re.UNICODE).strip("_") or "unknown_group"


def plot_group_decomposition(group, sample_index, score_position, group_model):
    scores_group, emission_group, excitation_group = normalize_group_cp(
        group_model["best"]["cp"], scale
    )
    component_count = scores_group.shape[1]
    component_names_group = [f"C{i + 1}" for i in range(component_count)]
    component_maps_full = np.einsum(
        "jf,kf->fjk", emission_group, excitation_group
    )
    contributions_full = scores_group[
        score_position, :, None, None
    ] * component_maps_full
    observed = np.maximum(np.asarray(eems[sample_index], dtype=float), 0)
    reconstructed = contributions_full.sum(axis=0)
    residual = observed - reconstructed
    observed_plot = observed
    reconstructed_plot = reconstructed
    residual_plot = residual
    contribution_plots = [
        contributions_full[component] for component in range(component_count)
    ]
    maps = [observed_plot, reconstructed_plot, residual_plot, *contribution_plots]
    titles = [
        "EEM gốc",
        "EEM tái tạo",
        "Phần dư",
        *[
            f"{name} (Fmax={scores_group[score_position, i]:.3g})"
            for i, name in enumerate(component_names_group)
        ],
    ]
    columns = 3
    rows = math.ceil(len(maps) / columns)
    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(5.2 * columns, 4.2 * rows),
        squeeze=False,
        constrained_layout=True,
    )
    intensity_values = [value for i, value in enumerate(maps) if i != 2]
    vmax = max(
        [float(np.nanmax(value)) for value in intensity_values if np.isfinite(value).any()]
        + [1e-12]
    )
    residual_max = max(float(np.nanmax(np.abs(residual_plot))), 1e-12)
    intensity_image = None
    residual_image = None
    intensity_axes = []
    for panel, (title, values) in enumerate(zip(titles, maps)):
        ax = axes.flat[panel]
        if panel == 2:
            residual_image = ax.pcolormesh(
                ex, em, values, shading="auto", cmap="RdBu_r",
                vmin=-residual_max, vmax=residual_max,
            )
        else:
            intensity_image = ax.pcolormesh(
                ex, em, values, shading="auto", cmap="viridis", vmin=0, vmax=vmax,
            )
            intensity_axes.append(ax)
        ax.set(title=title, xlabel="Ex (nm)", ylabel="Em (nm)")
    for ax in axes.flat[len(maps):]:
        ax.set_visible(False)
    fig.colorbar(intensity_image, ax=intensity_axes, label="Intensity", shrink=0.8)
    fig.colorbar(residual_image, ax=axes.flat[2], label="Residual", shrink=0.8)
    sample_id = samples.iloc[sample_index]["sample_id"]
    fig.suptitle(
        f"PARAFAC theo nhóm — {group} | EEM index {sample_index} | {sample_id} | rank {component_count}",
        fontsize=14,
    )
    output_path = GROUP_DECOMP_OUTPUT / (
        f"group_{group_file_name(group)}_eem-{sample_index}_decomposition.png"
    )
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    if SHOW_GROUP_DECOMPOSITIONS:
        display(fig)
    plt.close(fig)
    return output_path


if not globals().get("FIT_BY_GROUP", False):
    print("FIT_BY_GROUP=False; bỏ qua visualization theo nhóm.")
elif "group_models" not in globals():
    raise RuntimeError("Hãy chạy cell pipeline PARAFAC các nhóm theo nhóm trước cell này.")
else:
    group_plot_rows = []
    for group in selected_group_names:
        if group not in group_models:
            continue
        group_group = (
            samples.loc[samples["Group"].eq(group)]
            .sort_values(["sample_id", "Year", "Month"], kind="mergesort")
        )
        sample_index = int(group_group.index[0])
        group_indices = samples.index[samples["Group"].eq(group)].to_numpy(dtype=int)
        score_position = int(np.flatnonzero(group_indices == sample_index)[0])
        output_path = plot_group_decomposition(
            group, sample_index, score_position, group_models[group]
        )
        group_plot_rows.append({
            "Group": group,
            "sample_index": sample_index,
            "sample_id": samples.iloc[sample_index]["sample_id"],
            "selected_rank": int(group_models[group]["rank"]),
            "file": str(output_path),
        })
    group_plot_manifest = pd.DataFrame(group_plot_rows)
    group_plot_manifest.to_csv(
        GROUP_DECOMP_OUTPUT / "group_plot_manifest.csv", index=False
    )
    display(group_plot_manifest)
    print(f"Đã lưu {len(group_plot_manifest)} hình tại: {GROUP_DECOMP_OUTPUT}")

In [ ]:
# CV fits PARAFAC independently on each training fold.
import sys
sys.path.insert(0, str(ROOT / "scripts/diagnostics"))
from evaluate_parafac_all_components import run_all_components

# Prespecified rank; do not use ranks selected from full-group EEMs.
PARAFAC_CV_RANK = 4
all_component_scores = run_all_components(
    GROUP_OUTPUT, rank=PARAFAC_CV_RANK, include_single=True)
display(pd.read_csv(GROUP_OUTPUT / "group_best_component_xgb_r2.csv"))


## Linear Regression R² cross-validation theo nhóm

Cell này đánh giá hồi quy tuyến tính từ `max_*` của từng component. Component tốt nhất của mỗi nhóm và target được chọn theo `R2_linear_cv_mean`; `R2_linear_train` chỉ là điểm in-sample của cùng component. Background không được dùng trong phần V0.


In [ ]:
# Linear CV was evaluated together with XGBoost using the same train-only fits.
display(pd.read_csv(GROUP_OUTPUT / "group_best_component_linear_cv_r2.csv"))
